In [ ]:
#MOUNTING DRIVE
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
print(torch.cuda.get_device_name(0))

In [ ]:
#LOADING THE DATASET
from datasets import load_dataset
imdb_dataset = load_dataset("imdb")

In [ ]:
#SAVING THE DATASET TO DRIVE
imdb_dataset.save_to_disk("/content/drive/MyDrive/imdb_dataset")

In [ ]:
import os
os.listdir("/content/drive/MyDrive/imdb_dataset")

In [ ]:
from datasets import load_from_disk
dataset = load_from_disk("/content/drive/MyDrive/imdb_dataset")

In [ ]:
print(dataset)#PRINTING THE CONTENTS OF DATASET

#DATA UNDERSTANDING

In [ ]:
print(dataset.keys())

In [ ]:
#EXTRACTING ONLY THE TRAIN DATASET AND TEST DATASET
train_dataset = dataset['train']
test_dataset = dataset['test']
print(train_dataset.features)
print(test_dataset.features)

In [ ]:
print(train_dataset[0])#PRINTING ONLY THE FIRST ELEMENT OF TRAIN DATASET

In [ ]:
print(train_dataset[0]['text'])
print(train_dataset[0]['label'])

In [ ]:
print(train_dataset.features['text'])
print(train_dataset.features['label'])

In [ ]:
for i in range(10):#PRINTING THE FIRST 10 ELEMENTS OF THE TRAIN DATASET
  print(train_dataset[i]['text'])
  print(train_dataset[i]['label'])

In [ ]:
for i in range(5):# CHECKING THE LENGTH OF TOP 5 REVIEWS
  review = train_dataset[i]['text']
  print(f'Review length of {i} : {len(review)}')


In [ ]:
positive_count = 0
negative_count = 0
for rev in train_dataset:  #CHECKING THE COUNT OF TOTAL POSITIVE AND NEGATIVE REVIEWS
  if rev['label'] == 1:
    positive_count +=1
  else:
    negative_count +=1

print("Positive review counts", positive_count)
print("Negative review counts", negative_count)

#DATA PREPROCESSING

In [ ]:
from sklearn.model_selection import train_test_split
# SPLITTING THE TRAIN DATASET FOR TRAINING AND VALIDATION
x = list(train_dataset["text"])
y = list(train_dataset["label"])

train_text, val_text, train_label, val_label = train_test_split(x, y, test_size = 0.2, random_state = 42, stratify = y)

print('Train size:', len(train_text))
print('Validation size:', len(val_text))

In [ ]:
test_text = list(test_dataset['text'])
test_label = list(test_dataset['label'])


In [ ]:
type(train_label[0])

In [ ]:
#TEXT PREPROCESSING
from bs4 import BeautifulSoup
import re
#DEFINING METHOD FOR TEXT PREPROCESSING
def preprocessing(text):
  text = text.lower()#LOWERING THE TEXT
  text = BeautifulSoup(text, 'html.parser').get_text()#REMOVING HTML HAST TAGS
  text = re.sub(r'\s+',' ', text).strip()#REMOVING MULTIPLE SPACES
  return {'clean_text': text }


In [ ]:
#PASSING THE TEXT DATA INTO THE PREPROCESSING METHOD
train_text = [preprocessing(text) for text in train_text]
val_text = [preprocessing(text) for text in val_text]
test_text = [preprocessing(text) for text in test_text]
print('Train dataset',train_text)
print('Validation dataset', val_text)
print('Test dataset', test_text)

In [ ]:
train_texts = [sample['clean_text'] for sample in train_text]
val_texts = [sample['clean_text'] for sample in val_text]
test_texts = [sample['clean_text'] for sample in test_text]

In [ ]:
print(train_texts[0])

In [ ]:
print(test_texts[0])

In [ ]:
for i in range(5):
  print(train_texts[i])

In [ ]:
from collections import Counter

MAX_VOCAB = 20000
MAX_LEN = 150

#Tokenization
def tokenize(text):
  return text.split()

train_tokens = [tokenize(i) for i in train_texts]
val_tokens = [tokenize(i) for i in val_texts]

In [ ]:
#Vocabulary building
counter = Counter()

for tokens in train_tokens:
  counter.update(tokens)

vocab = {"<PAD>":0, "<UNK>":1}

for i, (word,_) in enumerate(counter.most_common(MAX_VOCAB-2)):
  vocab[word] = i + 2
print("Vocabulary Size:", len(vocab))

In [ ]:
#Encoding + Padding

def encode(tokens):

  encoded = [vocab.get(word, vocab["<UNK>"]) for word in tokens]
  #Truncating
  encoded = encoded[:MAX_LEN]
  #Padding
  encoded +=[vocab["<PAD>"]] * (MAX_LEN - len(encoded))
  return encoded

train_encoded = [encode(i) for i in train_tokens]
val_encoded = [encode(i) for i in val_tokens]

In [ ]:
# Tokenization
test_tokens = [tokenize(t) for t in test_texts]

# Encoding + Padding (using TRAIN vocab)
test_encoded = [encode(t) for t in test_tokens]



In [ ]:
#CONVERTING THE TEXT DATA INTO TENSOR
train_encoded = torch.tensor(train_encoded, dtype=torch.long)
train_label = torch.tensor(train_label, dtype=torch.long)

val_encoded = torch.tensor(val_encoded, dtype=torch.long)
val_label = torch.tensor(val_label, dtype=torch.long)

In [ ]:
test_encoded = torch.tensor(test_encoded, dtype=torch.long)
test_label = torch.tensor(test_label, dtype=torch.long)

In [ ]:
#CREATING DATA LOADER
import torch
from torch.utils.data import Dataset, TensorDataset, DataLoader

class IMDBDataset(Dataset):

    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

In [ ]:
train_data = IMDBDataset(train_encoded, train_label)

val_data = IMDBDataset(val_encoded, val_label)

test_data = IMDBDataset(test_encoded, test_label)

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(train_data, batch_size = BATCH_SIZE, shuffle = True)
val_loader = DataLoader(val_data, batch_size = BATCH_SIZE, shuffle = False)
test_loader = DataLoader(test_data, batch_size = BATCH_SIZE, shuffle = False)


In [ ]:
for i, j in test_loader:
  print(i.shape)
  print(j.shape)
  break

#CUSTOM LSTM MODEL BUILDING

In [ ]:
import torch
import torch.nn as nn

class CustomLSTM(nn.Module):

    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128):

        super(CustomLSTM, self).__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):

        embedded = self.embedding(x)

        lstm_out, (hidden, cell) = self.lstm(embedded)

        out = hidden[-1]

        logits = self.fc(out)

        return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CustomLSTM(vocab_size=len(vocab)).to(device)

In [ ]:
import torch.optim as optim
#OPTIMIZER
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
#DEFINING THE TRAINING LOOP
def train_one_epoch(model, loader, optimizer, criterion):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for X, y in loader:

        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()

        outputs = model(X)

        loss = criterion(outputs, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        # accuracy
        preds = torch.argmax(outputs, dim=1)

        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss / len(loader), correct / total

In [ ]:
#DEFINING THE VALIDATION LOOP
def validation(model, loader, criterion):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for X, y in loader:

            X, y = X.to(device), y.to(device)

            outputs = model(X)

            loss = criterion(outputs, y)

            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss / len(loader), correct / total

In [ ]:
#DEFINING EPOCHS AND TRIANING THE MODEL
EPOCHS = 5

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion)

    val_loss, val_acc = validation(
        model, val_loader, criterion)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print()
#SAVING THE MODEL
import os
import torch
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')
save_dir = "/content/drive/MyDrive/lstm_imdb_model"
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, "Custom_lstm_sentiment_analysis_model.pth")

torch.save(model.state_dict(), model_path)

print("Model saved at:", model_path)

#CUSTOM LSTM MODEL EVALUATION

In [ ]:
import torch

def evaluate_test(model, test_loader, criterion):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for X, y in test_loader:

            X, y = X.to(device), y.to(device)

            outputs = model(X)

            loss = criterion(outputs, y)

            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

            # store for metrics
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    accuracy = correct / total

    return total_loss / len(test_loader), accuracy, all_preds, all_labels

CUSTOM LSTM RESULTS

In [ ]:
test_loss, test_acc, y_pred, y_true = evaluate_test(
    model,
    test_loader,
    criterion
)
import torch
import os

save_dir = "/content/drive/MyDrive/lstm_imdb_model"
os.makedirs(save_dir, exist_ok=True)

results_path = os.path.join(save_dir, "CustomLSTM_test_results.pth")

results = {
    "test_loss": test_loss,
    "test_acc": test_acc,
    "y_pred": y_pred,
    "y_true": y_true
}

torch.save(results, results_path)

print("Results saved at:", results_path)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, target_names=["Negative", "Positive"]))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Negative", "Positive"],
            yticklabels=["Negative", "Positive"])

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

#PRETRAINED AWD-LSTM (ULMFiT) MODEL


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
from datasets import load_dataset
dataset = load_dataset("imdb")#LOADING THE DATASET

print(dataset)

In [ ]:
import pandas as pd
#CREATING THE DICTIONARY DATA TO DATAFRAME
train_df = pd.DataFrame({
    "text": dataset["train"]["text"],
    "label": dataset["train"]["label"]
})

test_df = pd.DataFrame({
    "text": dataset["test"]["text"],
    "label": dataset["test"]["label"]
})

In [ ]:
#DATA PREPROCESSING
from bs4 import BeautifulSoup
import re

def clean_text(text):
    text = BeautifulSoup(text, "html.parser").get_text()  # remove HTML
    text = re.sub(r'\s+', ' ', text).strip()              # normalize spaces
    return text

train_df["text"] = train_df["text"].apply(clean_text)
test_df["text"] = test_df["text"].apply(clean_text)

In [ ]:
from fastai.text.all import *
#CREATING DATALOADERS FASTAI WAY
dls_lm = TextDataLoaders.from_df(
    train_df,
    text_col="text",
    is_lm=True,
    valid_pct=0.1,
    seed=42,
    bs=64
)

In [ ]:
#Language Model Training
learn_lm = language_model_learner(
    dls_lm,
    AWD_LSTM,
    drop_mult=0.5,
    metrics=[accuracy, Perplexity()]
)

learn_lm.fit_one_cycle(1, 2e-2)

In [ ]:
learn_lm.save_encoder("ulmfit_encoder")

In [ ]:
#Text Classifier Training
learn_clas = text_classifier_learner(
    dls_clas,
    AWD_LSTM,
    drop_mult=0.5,
    metrics=accuracy
)

In [ ]:
learn_clas = learn_clas.load_encoder("ulmfit_encoder")

In [ ]:
#TRAINING ULMFiT TEXT CLASSIFICATION MODEL
learn_clas.fit_one_cycle(1, 2e-2)

In [ ]:
learn_clas.export("ulmfit_classifier_sentiment_analysis_model_export.pkl")

In [ ]:
#TESTING
test_dl = learn_clas.dls.test_dl(test_df, with_labels=True)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Get predictions from model
preds, targs = learn_clas.get_preds(dl=test_dl)

# Predicted labels
pred_labels = preds.argmax(dim=1).numpy()

# True labels
true_labels = targs.numpy()

# Accuracy
test_accuracy = accuracy_score(true_labels, pred_labels)

print("Test Accuracy:", test_accuracy)

# Classification report
print(classification_report(true_labels, pred_labels))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
save_path = "/content/drive/MyDrive/ULMFIT_Sentiment_Model"

#SAVING THE MODEL AND ENCODER
import os
os.makedirs(save_path, exist_ok=True)
learn_clas.save_encoder(f"{save_path}/ulmfit_encoder")
learn_clas.export(f"{save_path}/ulmfit_classifier_sentiment_analysis_model_export.pkl")

In [ ]:
#SAVING THE RESULTS
import pandas as pd

# Predicted labels
pred_labels = preds.argmax(dim=1).numpy()

# True labels from test dataframe
true_labels = test_df["label"].values

# Create results dataframe
results_df = pd.DataFrame({
    "text": test_df["text"],
    "true_label": true_labels,
    "pred_label": pred_labels
})

In [ ]:
results_df.to_csv("/content/drive/MyDrive/ulmfit_results.csv", index=False)

In [ ]:
with open("/content/drive/MyDrive/ulmfit_accuracy.txt", "w") as f:
    f.write(f"Test Accuracy: {test_accuracy}")